# 1부두 전처리

## 1. 파일 읽기 → 원본 확인

In [45]:
from pathlib import Path
import pandas as pd
from IPython.display import display
DATA = Path('../data/old')
def read_csv(path):
    for encoding in ['utf-8-sig', 'cp949', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=encoding, dtype=str).fillna('')
        except UnicodeDecodeError:
            continue
    raise ValueError('파일 인코딩을 확인하세요.')
record = read_csv(DATA / 'record_except_168.csv')
port1 = read_csv(DATA / 'port1.csv')
print('record:', record.shape, 'port1:', port1.shape)
display(record.head(5))
display(port1.head(5))

record: (19981, 13) port1: (2928, 12)


,호출부호,선명,입항횟수,총톤수,입항일시,출항일시,CIQ수속일자,수리일시,MRN 번호,계선장소,차항지,전출항지,재항시간
0,3FRF8,SONG YUN HE,2022.0,"16,737",2022-01-01 00:48:00,2022-01-03 01:52:00,2022-01-03 1:52,2022-01-03 15:58,22COHES152E,신항 5부두 3선석,QINGDAO,광양,49.06666666666667
1,VRNA9,SEASPAN THAMES,2022.0,"112,967",2022-01-01 02:00:00,2022-01-02 17:43:00,2022-01-02 17:43,2022-01-03 11:18,22HLCU4202E,신항 3부두 4선석,VANCOUVER,SHANGHAI,39.71666666666667
2,VRRP8,CMA CGM MUMBAI,2022.0,"112,967",2022-01-01 03:00:00,2022-01-02 04:06:00,2022-01-02 4:00,2022-01-03 11:22,22CMAL0030E,신항 5부두 2선석,MANZANILLO,SHANGHAI,25.1
3,D5TL4,MSC JEWEL,2022.0,"140,976",2022-01-01 03:22:00,2022-01-03 05:00:00,2022-01-03 5:00,2022-01-03 11:19,22MSCU0004E,신항 1부두 3선석,MANZANILLO,HONG KONG,49.63333333333333
4,A8YT7,MALIAKOS,2022.0,"41,391",2022-01-01 03:35:00,2022-01-01 16:45:00,2022-01-01 16:00,2022-01-03 11:07,22SMLME016E,신항 2부두 1선석,VANCOUVER,SHANGHAI,13.166666666666666


,선석,선사,모선항차,선사항차,선명,ROUTE,반입마감시한,접안(예정)일시,출항(예정)일시,양하,적하,Shift
0,T3(P),MSC,MJEW008,FA147A/FA147A,MSC JEWEL,ASPA2E,2021-12-31 16:00,2022-01-01 3:40,2022-01-03 5:00,1263,2905,2
1,T1(P),COS,COSP001,090E/090E,COSCO SAO PAULO,WSA3,2021-12-31 20:00,2022-01-01 7:20,2022-01-02 7:00,402,1066,0
2,T2(P),HLC,COPX001,2140W/2152E,COPIAPO,AN1,2022-01-01 22:00,2022-01-02 8:10,2022-01-04 12:00,1471,1998,108
3,T1(P),MSC,MELN003,UK150A/UK150A,MSC ELAINE,CHINK,2022-01-01 22:00,2022-01-02 9:10,2022-01-03 20:00,838,1281,2
4,T3(P),MSC,MDES002,FI144R/FI150A,MSC DESIREE,AS2,2022-01-02 19:00,2022-01-03 7:20,2022-01-04 20:00,1139,1900,570


## 2. 1부두와 분석 기간 추출 → 건수 확인

In [46]:
r = record[record['계선장소'].str.contains('신항 1부두')].copy()
r['rid'] = r.index
r['입항'] = pd.to_datetime(r['입항일시'])
r['출항'] = pd.to_datetime(r['출항일시'])
r = r[r['입항'].dt.year.between(2022,2025)].copy()
p = port1.copy()
p['pid'] = p.index
p['접안'] = pd.to_datetime(p['접안(예정)일시'])
p['터미널출항'] = pd.to_datetime(p['출항(예정)일시'])
p = p[p['접안'].dt.year.between(2022,2025)].copy()
print('record 1부두:', len(r), 'port1 기간 내:', len(p))
display(r[['선명','입항','출항','계선장소']].head(5))

record 1부두: 2920 port1 기간 내: 2928


,선명,입항,출항,계선장소
3,MSC JEWEL,2022-01-01 03:22:00,2022-01-03 05:00:00,신항 1부두 3선석
7,COSCO SAO PAULO,2022-01-01 07:10:00,2022-01-02 06:55:00,신항 1부두 1선석
17,COPIAPO,2022-01-02 08:15:00,2022-01-04 12:05:00,신항 1부두 2선석
18,MSC ELAINE,2022-01-02 08:58:00,2022-01-03 19:55:00,신항 1부두 1선석
30,MSC DESIREE,2022-01-03 07:10:00,2022-01-04 20:20:00,신항 1부두 3선석


## 3. 데이터 기초통계 확인

In [47]:
display(record.shape)
record.info()
display(record.columns.to_list())
display(record.index)
display(record.dtypes)
display(record.isna().sum())
display(record.eq('').sum())
display(record.describe())

(19981, 13)

<class 'pandas.DataFrame'>
RangeIndex: 19981 entries, 0 to 19980
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   호출부호     19981 non-null  str  
 1   선명       19981 non-null  str  
 2   입항횟수     19981 non-null  str  
 3   총톤수      19981 non-null  str  
 4   입항일시     19981 non-null  str  
 5   출항일시     19981 non-null  str  
 6   CIQ수속일자  19981 non-null  str  
 7   수리일시     19981 non-null  str  
 8   MRN 번호   19981 non-null  str  
 9   계선장소     19981 non-null  str  
 10  차항지      19981 non-null  str  
 11  전출항지     19981 non-null  str  
 12  재항시간     19981 non-null  str  
dtypes: str(13)
memory usage: 5.1 MB


['호출부호',
 '선명',
 '입항횟수',
 '총톤수',
 '입항일시',
 '출항일시',
 'CIQ수속일자',
 '수리일시',
 'MRN 번호',
 '계선장소',
 '차항지',
 '전출항지',
 '재항시간']

RangeIndex(start=0, stop=19981, step=1)

호출부호       str
선명         str
입항횟수       str
총톤수        str
입항일시       str
출항일시       str
CIQ수속일자    str
수리일시       str
MRN 번호     str
계선장소       str
차항지        str
전출항지       str
재항시간       str
dtype: object

호출부호       0
선명         0
입항횟수       0
총톤수        0
입항일시       0
출항일시       0
CIQ수속일자    0
수리일시       0
MRN 번호     0
계선장소       0
차항지        0
전출항지       0
재항시간       0
dtype: int64

호출부호       0
선명         0
입항횟수       0
총톤수        0
입항일시       0
출항일시       0
CIQ수속일자    0
수리일시       0
MRN 번호     0
계선장소       0
차항지        0
전출항지       0
재항시간       0
dtype: int64

,호출부호,선명,입항횟수,총톤수,입항일시,출항일시,CIQ수속일자,수리일시,MRN 번호,계선장소,차항지,전출항지,재항시간
count,19981,19981,19981,19981,19981,19981,19981,19981,19981,19981,19981,19981,19981
unique,2487,2421,4,850,19726,19690,17828,7624,19981,20,174,203,3379
top,C6TN2,MAGNA,2023.0,"5,403",2022-02-07 18:00:00,2022-07-22 07:00:00,2022-01-27 6:00,2022-09-13 10:18,22COHES152E,신항 2부두 1선석,SHANGHAI,QINGDAO,17.0
freq,363,363,5287,481,3,3,5,12,1,1625,2472,2610,62


In [48]:
display(port1.shape)
port1.info()
display(port1.columns.to_list())
display(port1.index)
display(port1.dtypes)
display(port1.isna().sum())
display(port1.eq('').sum())
display(port1.describe())

(2928, 12)

<class 'pandas.DataFrame'>
RangeIndex: 2928 entries, 0 to 2927
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   선석        2928 non-null   str  
 1   선사        2928 non-null   str  
 2   모선항차      2928 non-null   str  
 3   선사항차      2928 non-null   str  
 4   선명        2928 non-null   str  
 5   ROUTE     2928 non-null   str  
 6   반입마감시한    2928 non-null   str  
 7   접안(예정)일시  2928 non-null   str  
 8   출항(예정)일시  2928 non-null   str  
 9   양하        2928 non-null   str  
 10  적하        2928 non-null   str  
 11  Shift     2928 non-null   str  
dtypes: str(12)
memory usage: 557.6 KB


['선석',
 '선사',
 '모선항차',
 '선사항차',
 '선명',
 'ROUTE',
 '반입마감시한',
 '접안(예정)일시',
 '출항(예정)일시',
 '양하',
 '적하',
 'Shift']

RangeIndex(start=0, stop=2928, step=1)

선석          str
선사          str
모선항차        str
선사항차        str
선명          str
ROUTE       str
반입마감시한      str
접안(예정)일시    str
출항(예정)일시    str
양하          str
적하          str
Shift       str
dtype: object

선석          0
선사          0
모선항차        0
선사항차        0
선명          0
ROUTE       0
반입마감시한      0
접안(예정)일시    0
출항(예정)일시    0
양하          0
적하          0
Shift       0
dtype: int64

선석          0
선사          0
모선항차        0
선사항차        0
선명          0
ROUTE       0
반입마감시한      9
접안(예정)일시    0
출항(예정)일시    0
양하          0
적하          0
Shift       0
dtype: int64

,선석,선사,모선항차,선사항차,선명,ROUTE,반입마감시한,접안(예정)일시,출항(예정)일시,양하,적하,Shift
count,2928,2928,2928,2928,2928,2928,2928,2928,2928,2928,2928,2928
unique,6,22,2111,2728,629,60,2780,2922,2854,1627,1838,365
top,T1(P),MSC,MDEI001,001E/001E,MSC GIANNINA II,AN1,,2022-09-11 8:20,2023-12-30 2:00,0,0,0
freq,946,1973,4,8,66,211,9,3,3,141,100,1269


## 4. 데이터 연결방식 비교 1 — 선박명 + 출항시각
공백·기호·대소문자만 정리.
같은 이름의 모든 방문이 후보로 병합되므로 시간 조건을 적용해야 합니다.
출항 차이 6시간 이내에서 가까운 후보부터 선택합니다.

In [49]:
r['연결선명'] = r['선명'].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
p['연결선명'] = p['선명'].str.upper().str.replace(r'[^A-Z0-9]', '', regex=True)
display(r[['선명','연결선명']].drop_duplicates().head(5))

,선명,연결선명
3,MSC JEWEL,MSCJEWEL
7,COSCO SAO PAULO,COSCOSAOPAULO
17,COPIAPO,COPIAPO
18,MSC ELAINE,MSCELAINE
30,MSC DESIREE,MSCDESIREE


In [50]:
비교 = r.merge(p, on='연결선명', suffixes=('_record','_port1'))
비교['출항차이'] = (비교['출항']-비교['터미널출항']).abs()
비교['입항차이'] = (비교['입항']-비교['접안']).abs()
확인열 = ['rid','pid','선명_record','선명_port1','모선항차','입항','접안','출항','터미널출항','입항차이','출항차이']
display(비교[확인열].head(5))

,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이
0,3,0,MSC JEWEL,MSC JEWEL,MJEW008,2022-01-01 03:22:00,2022-01-01 03:40:00,2022-01-03 05:00:00,2022-01-03 05:00:00,0 days 00:18:00,0 days 00:00:00
1,3,160,MSC JEWEL,MSC JEWEL,MJEW001,2022-01-01 03:22:00,2022-03-17 12:15:00,2022-01-03 05:00:00,2022-03-18 17:00:00,75 days 08:53:00,74 days 12:00:00
2,3,213,MSC JEWEL,MSC JEWEL,MJEW002,2022-01-01 03:22:00,2022-04-11 13:40:00,2022-01-03 05:00:00,2022-04-13 16:00:00,100 days 10:18:00,100 days 11:00:00
3,3,407,MSC JEWEL,MSC JEWEL,MJEW004,2022-01-01 03:22:00,2022-07-22 09:30:00,2022-01-03 05:00:00,2022-07-24 15:00:00,202 days 06:08:00,202 days 10:00:00
4,3,550,MSC JEWEL,MSC JEWEL,MJEW005,2022-01-01 03:22:00,2022-10-09 14:20:00,2022-01-03 05:00:00,2022-10-10 11:00:00,281 days 10:58:00,280 days 06:00:00


In [51]:
출항연결 = (비교[비교['출항차이']<=pd.Timedelta(hours=6)]
    .sort_values(['출항차이','입항차이','rid','pid'])
    .drop_duplicates('rid').drop_duplicates('pid').copy())
출항연결_이름수정전 = 출항연결.copy()
print('이름 수정 전 연결:',len(출항연결))
display(출항연결[확인열].head(5))

이름 수정 전 연결: 2893


,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이
412,353,58,MSC MAKOTO II,MSC MAKOTO II,MMKT003,2022-01-27 05:00:00,2022-01-27 05:00:00,2022-01-27 12:00:00,2022-01-27 12:00:00,0 days,0 days
559,497,82,FLEUR N,FLEUR N,FLEX001,2022-02-07 16:00:00,2022-02-07 16:00:00,2022-02-08 09:00:00,2022-02-08 09:00:00,0 days,0 days
4518,3672,534,MSC EMMA,MSC EMMA,MSEM006,2022-10-01 17:00:00,2022-10-01 17:00:00,2022-10-03 08:00:00,2022-10-03 08:00:00,0 days,0 days
6608,5017,719,MSC CARPATHIA III,MSC CARPATHIA III,MCPZ001,2023-01-05 16:00:00,2023-01-05 16:00:00,2023-01-06 08:00:00,2023-01-06 08:00:00,0 days,0 days
6620,5028,720,MSC SHRISTI,MSC SHRISTI,MSHR002,2023-01-06 10:10:00,2023-01-06 10:10:00,2023-01-07 04:00:00,2023-01-07 04:00:00,0 days,0 days


## 5. 미연결 기록 확인 → 이름 표기 차이 검토
이 단계에서 바로 제외하지 않습니다. 가까운 날짜에 이름이 다른 후보가 있는지 확인합니다. 날짜가 비슷하다고 자동으로 동일 선박으로 확정하지 않습니다.

In [52]:
미연결_record=r[~r['rid'].isin(출항연결['rid'])].copy()
미연결_port1=p[~p['pid'].isin(출항연결['pid'])].copy()
print('record 미연결:',len(미연결_record),'port1 미연결:',len(미연결_port1))
display(미연결_record[['rid','선명','입항','출항','계선장소']])
display(미연결_port1[['pid','선명','모선항차','접안','터미널출항','선석']])

record 미연결: 27 port1 미연결: 35


,rid,선명,입항,출항,계선장소
205,205,MSC KYMEA,2022-01-16 07:50:00,2022-01-17 16:06:00,신항 1부두 3선석
393,393,MSC KYMEA,2022-01-30 02:25:00,2022-01-30 21:45:00,신항 1부두 1선석
503,503,MSC LISBON,2022-02-07 22:00:00,2022-02-08 11:56:00,신항 1부두 2선석
662,662,MONTEVIDEO EXPRESS,2022-02-20 04:20:00,2022-02-22 06:57:00,신항 1부두 2선석
700,700,MSC KYMEA,2022-02-22 17:10:00,2022-02-23 04:08:00,신항 1부두 1선석
734,734,TAYMA,2022-02-24 20:55:00,2022-02-26 15:10:00,신항 1부두 3선석
1702,1702,TAYMA,2022-05-06 00:30:00,2022-05-06 23:54:00,신항 1부두 2선석
1757,1757,PACIFIC BUSAN,2022-05-10 03:13:00,2022-05-11 04:00:00,신항 1부두 2선석
2156,2156,MSC TAMPICO,2022-06-07 12:50:00,2022-06-08 18:52:00,신항 1부두 2선석
3321,3321,MSC TAMPICO,2022-09-03 02:04:00,2022-09-03 23:57:00,신항 1부두 2선석


,pid,선명,모선항차,접안,터미널출항,선석
34,34,MSC KYMEA II,MKYM002,2022-01-17 01:40:00,2022-01-17 16:00:00,T3(P)
66,66,MSC KYMEA II,MKYM003,2022-01-30 02:30:00,2022-01-30 21:40:00,T1(S)
83,83,MSC LISBON IX,MVLS001,2022-02-07 22:20:00,2022-02-08 12:00:00,T2(P)
109,109,NAVEGANTES EXPRESS,MVEX001,2022-02-20 05:00:00,2022-02-22 07:00:00,T2(P)
115,115,MSC KYMEA II,MKYM005,2022-02-22 17:30:00,2022-02-23 04:00:00,T1(P)
119,119,TAYMA EXPRESS,TAYX001,2022-02-24 21:20:00,2022-02-26 15:00:00,T3(P)
135,135,XIN BIN ZHOU,MXBZ005,2022-03-05 20:20:00,2022-03-06 15:00:00,T1(P)
261,261,TAYMA EXPRESS,TAYX002,2022-05-06 00:50:00,2022-05-06 23:59:00,T2(P)
321,321,MSC TAMPICO V,MTMP001,2022-06-07 13:15:00,2022-06-08 19:00:00,T2(P)
384,384,MSC MUNDRA,MMDU001,2022-07-09 17:05:00,2022-07-10 17:00:00,T1(P)


In [53]:
이름검토=미연결_record.merge(미연결_port1,how='cross',suffixes=('_record','_port1'))
이름검토['출항차이']=(이름검토['출항']-이름검토['터미널출항']).abs()
이름검토=이름검토[이름검토['출항차이']<=pd.Timedelta(hours=6)]
display(이름검토[['선명_record','선명_port1','입항','접안','출항','터미널출항','출항차이']].sort_values(['선명_record','출항차이']))

,선명_record,선명_port1,입항,접안,출항,터미널출항,출항차이
108,MONTEVIDEO EXPRESS,NAVEGANTES EXPRESS,2022-02-20 04:20:00,2022-02-20 05:00:00,2022-02-22 06:57:00,2022-02-22 07:00:00,0 days 00:03:00
36,MSC KYMEA,MSC KYMEA II,2022-01-30 02:25:00,2022-01-30 02:30:00,2022-01-30 21:45:00,2022-01-30 21:40:00,0 days 00:05:00
0,MSC KYMEA,MSC KYMEA II,2022-01-16 07:50:00,2022-01-17 01:40:00,2022-01-17 16:06:00,2022-01-17 16:00:00,0 days 00:06:00
144,MSC KYMEA,MSC KYMEA II,2022-02-22 17:10:00,2022-02-22 17:30:00,2022-02-23 04:08:00,2022-02-23 04:00:00,0 days 00:08:00
72,MSC LISBON,MSC LISBON IX,2022-02-07 22:00:00,2022-02-07 22:20:00,2022-02-08 11:56:00,2022-02-08 12:00:00,0 days 00:04:00
944,MSC RANIA VIII,MSC RANIA,2025-11-22 18:19:00,2025-11-22 18:19:00,2025-11-24 00:01:00,2025-11-24 00:01:00,0 days 00:00:00
326,MSC TAMPICO,MSC TAMPICO V,2022-09-03 02:04:00,2022-09-03 02:25:00,2022-09-03 23:57:00,2022-09-04 00:00:00,0 days 00:03:00
288,MSC TAMPICO,MSC TAMPICO V,2022-06-07 12:50:00,2022-06-07 13:15:00,2022-06-08 18:52:00,2022-06-08 19:00:00,0 days 00:08:00
362,MSC TAMPICO,MSC TAMPICO V,2022-09-15 16:36:00,2022-09-15 16:45:00,2022-09-17 05:12:00,2022-09-17 05:00:00,0 days 00:12:00
217,TAYMA,TAYMA EXPRESS,2022-05-06 00:30:00,2022-05-06 00:50:00,2022-05-06 23:54:00,2022-05-06 23:59:00,0 days 00:05:00


### 5-1. 앞서 검토한 9개 이름 대응 적용
원본 선명은 유지하고 연결용 이름만 통일.
이름 변경 사전은 기존 검토 결과이며, 항차 목록을 직접 입력한 것이 아닙니다.
양쪽에 같은 규칙 적용.

In [54]:
선박명_변경목록 = {
'NAVEGANTESEXPRESS':'MONTEVIDEOEXPRESS',
'MSCKYMEAII':'MSCKYMEA', 'MSCLISBONIX':'MSCLISBON',
'MSCRANIA':'MSCRANIAVIII', 'MSCTAMPICOV':'MSCTAMPICO',
'TAYMAEXPRESS':'TAYMA', 'DANUBE':'ZIMDANUBE',
'GANGES':'ZIMGANGES', 'SPARROW':'ZIMSPARROW'}
display(pd.DataFrame(선박명_변경목록.items(),columns=['변경 전','통일 이름']))
r['연결선명']=r['연결선명'].replace(선박명_변경목록)
p['연결선명']=p['연결선명'].replace(선박명_변경목록)
비교 = r.merge(p, on='연결선명', suffixes=('_record','_port1'))
비교['출항차이'] = (비교['출항']-비교['터미널출항']).abs()
비교['입항차이'] = (비교['입항']-비교['접안']).abs()
출항연결 = (비교[비교['출항차이']<=pd.Timedelta(hours=6)]
    .sort_values(['출항차이','입항차이','rid','pid'])
    .drop_duplicates('rid').drop_duplicates('pid').copy())
print('이름 통일 후 연결:',len(출항연결))
display(출항연결[~출항연결['rid'].isin(출항연결_이름수정전['rid'])][확인열])

,변경 전,통일 이름
0,NAVEGANTESEXPRESS,MONTEVIDEOEXPRESS
1,MSCKYMEAII,MSCKYMEA
2,MSCLISBONIX,MSCLISBON
3,MSCRANIA,MSCRANIAVIII
4,MSCTAMPICOV,MSCTAMPICO
5,TAYMAEXPRESS,TAYMA
6,DANUBE,ZIMDANUBE
7,GANGES,ZIMGANGES
8,SPARROW,ZIMSPARROW


이름 통일 후 연결: 2911


,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이
33166,19464,2855,MSC RANIA VIII,MSC RANIA,MSRA001,2025-11-22 18:19:00,2025-11-22 18:19:00,2025-11-24 00:01:00,2025-11-24 00:01:00,0 days 00:00:00,0 days 00:00:00
19609,10966,1632,ZIM GANGES,GANGES,ZZB5001,2024-02-22 06:55:00,2024-02-22 07:00:00,2024-02-23 19:58:00,2024-02-23 20:00:00,0 days 00:05:00,0 days 00:02:00
15564,9176,1356,ZIM DANUBE,DANUBE,ZZB4001,2023-10-18 03:50:00,2023-10-18 04:15:00,2023-10-19 15:58:00,2023-10-19 16:00:00,0 days 00:25:00,0 days 00:02:00
16769,9711,1435,ZIM SPARROW,SPARROW,ZZS4001,2023-11-24 12:00:00,2023-11-24 12:30:00,2023-11-24 21:02:00,2023-11-24 21:00:00,0 days 00:30:00,0 days 00:02:00
17732,10079,1494,ZIM GANGES,GANGES,ZZB5001,2023-12-20 00:30:00,2023-12-20 01:00:00,2023-12-20 18:58:00,2023-12-20 19:00:00,0 days 00:30:00,0 days 00:02:00
4265,3321,489,MSC TAMPICO,MSC TAMPICO V,MTMP002,2022-09-03 02:04:00,2022-09-03 02:25:00,2022-09-03 23:57:00,2022-09-04 00:00:00,0 days 00:21:00,0 days 00:03:00
862,662,109,MONTEVIDEO EXPRESS,NAVEGANTES EXPRESS,MVEX001,2022-02-20 04:20:00,2022-02-20 05:00:00,2022-02-22 06:57:00,2022-02-22 07:00:00,0 days 00:40:00,0 days 00:03:00
653,503,83,MSC LISBON,MSC LISBON IX,MVLS001,2022-02-07 22:00:00,2022-02-07 22:20:00,2022-02-08 11:56:00,2022-02-08 12:00:00,0 days 00:20:00,0 days 00:04:00
519,393,66,MSC KYMEA,MSC KYMEA II,MKYM003,2022-01-30 02:25:00,2022-01-30 02:30:00,2022-01-30 21:45:00,2022-01-30 21:40:00,0 days 00:05:00,0 days 00:05:00
2110,1702,261,TAYMA,TAYMA EXPRESS,TAYX002,2022-05-06 00:30:00,2022-05-06 00:50:00,2022-05-06 23:54:00,2022-05-06 23:59:00,0 days 00:20:00,0 days 00:05:00


## 6. 데이터 연결방식 비교 2 — 입항↔접안 우선
출항 우선 결과를 유지하면서 입항 우선 결과와 비교.
기존 원본 기준 출항 우선 2,911건, 입항 우선 2,771건입니다.

In [55]:
입항연결=(비교[비교['입항차이']<=pd.Timedelta(hours=6)].sort_values(['입항차이','출항차이','rid','pid']).drop_duplicates('rid').drop_duplicates('pid').copy())
print('출항 우선:',len(출항연결),'입항 우선:',len(입항연결))
방식비교=출항연결[['rid','pid','선명_record','모선항차']].merge(입항연결[['rid','pid','모선항차']],on='rid',suffixes=('_출항우선','_입항우선'))
다른연결=방식비교[방식비교['pid_출항우선']!=방식비교['pid_입항우선']]
print('연결 대상 변경:',len(다른연결))
display(다른연결)
display(입항연결[입항연결['출항차이']>pd.Timedelta(hours=6)][확인열])

출항 우선: 2911 입항 우선: 2771
연결 대상 변경: 16


,rid,pid_출항우선,선명_record,모선항차_출항우선,pid_입항우선,모선항차_입항우선
897,5989,864,MSC ALDI III,MADI003,860,MADI002
898,19262,2834,MSC RICCARDA II,MRIA012,2830,MRIA011
899,17143,2549,MSC URSULA VI,MURS002,2543,MURS001
972,11032,1644,MSC SAVONA,MSVO002,1638,MSVO001
1125,853,143,XIN BIN ZHOU,MXBZ006,135,MXBZ005
1126,16926,2517,CAUQUENES,CQNX002,2511,CQNX001
1127,10119,1511,MSC AVNI,MAVN004,1503,MAVN003
1635,3014,452,MSC LA SPEZIA,MLPZ002,446,MLPZ001
1636,12253,1820,MSC MAGNUM VII,MMGG003,1813,MMGG002
1766,8946,1323,BOTANY,MBON005,1309,MBON004


,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이
28652,16314,2429,CAPTAIN THANASIS I,CAPTAIN THANASIS I,MCTI008,2025-03-27 06:50:00,2025-03-27 06:50:00,2025-03-28 21:43:00,2025-03-28 07:00:00,0 days 00:00:00,0 days 14:43:00
28114,15867,2371,MSC FOLEGANDROS VI,MSC FOLEGANDROS VI,MFGD002,2025-02-20 22:15:00,2025-02-20 22:15:00,2025-02-23 06:58:00,2025-02-22 05:07:00,0 days 00:00:00,1 days 01:51:00
32943,19262,2830,MSC RICCARDA II,MSC RICCARDA II,MRIA011,2025-11-08 05:00:00,2025-11-08 05:00:00,2025-11-11 04:57:00,2025-11-08 19:48:00,0 days 00:00:00,2 days 09:09:00
29783,17143,2543,MSC URSULA VI,MSC URSULA VI,MURS001,2025-05-29 20:42:00,2025-05-29 20:42:00,2025-06-03 11:06:00,2025-05-30 12:01:00,0 days 00:00:00,3 days 23:05:00
3119,2576,384,MSC MUNDRA,MSC MUNDRA,MMDU001,2022-07-09 17:00:00,2022-07-09 17:05:00,2022-07-15 11:52:00,2022-07-10 17:00:00,0 days 00:05:00,4 days 18:52:00
14997,9000,1317,SONG YUN HE,SONG YUN HE,CSYH020,2023-10-05 07:35:00,2023-10-05 07:30:00,2023-10-12 00:50:00,2023-10-05 18:00:00,0 days 00:05:00,6 days 06:50:00
25942,14496,2151,MSC APOLLO,MSC APOLLO,MPLL002,2024-11-07 07:25:00,2024-11-07 07:36:00,2024-11-10 12:53:00,2024-11-07 16:00:00,0 days 00:11:00,2 days 20:53:00
8489,5989,860,MSC ALDI III,MSC ALDI III,MADI002,2023-03-15 10:47:00,2023-03-15 11:00:00,2023-03-17 13:00:00,2023-03-16 01:00:00,0 days 00:13:00,1 days 12:00:00
19684,11032,1638,MSC SAVONA,MSC SAVONA,MSVO001,2024-02-27 01:47:00,2024-02-27 02:00:00,2024-03-01 23:59:00,2024-02-27 12:00:00,0 days 00:13:00,3 days 11:59:00
3758,3014,446,MSC LA SPEZIA,MSC LA SPEZIA,MLPZ001,2022-08-11 00:15:00,2022-08-11 00:30:00,2022-08-14 11:55:00,2022-08-12 00:00:00,0 days 00:15:00,2 days 11:55:00


### 6-1. 출항 차이가 크지만 입항↔접안이 같은 별도 확인 사례
MSC FOLEGANDROS VI 1건(확인용).

In [56]:
별도연결=비교[(~비교['rid'].isin(출항연결['rid'])) & (~비교['pid'].isin(출항연결['pid'])) & (비교['입항차이']==pd.Timedelta(0))].copy()
print('별도 확인:',len(별도연결))
display(별도연결[확인열])

별도 확인: 1


,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이
28114,15867,2371,MSC FOLEGANDROS VI,MSC FOLEGANDROS VI,MFGD002,2025-02-20 22:15:00,2025-02-20 22:15:00,2025-02-23 06:58:00,2025-02-22 05:07:00,0 days,1 days 01:51:00


## 7. 한 record에 여러 port1 항차가 포함되는 경우 확인
완전 포함과 경계 30분 허용을 각각 계산합니다.
기존 원본에서 8건과 16건입니다. 미대응 8건과는 다른 개념입니다.

In [57]:
완전포함=비교[(비교['접안']>=비교['입항']) & (비교['터미널출항']<=비교['출항']) & (비교['터미널출항']>비교['접안'])].copy()
완전포함수=완전포함.groupby('rid')['pid'].nunique()
print('완전 포함 2항차 이상:',(완전포함수>=2).sum())
display(완전포함[완전포함['rid'].isin(완전포함수[완전포함수>=2].index)][확인열])

완전 포함 2항차 이상: 8


,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이
8489,5989,860,MSC ALDI III,MSC ALDI III,MADI002,2023-03-15 10:47:00,2023-03-15 11:00:00,2023-03-17 13:00:00,2023-03-16 01:00:00,0 days 00:13:00,1 days 12:00:00
8490,5989,864,MSC ALDI III,MSC ALDI III,MADI003,2023-03-15 10:47:00,2023-03-16 20:40:00,2023-03-17 13:00:00,2023-03-17 13:00:00,1 days 09:53:00,0 days 00:00:00
17794,10119,1503,MSC AVNI,MSC AVNI,MAVN003,2023-12-23 07:15:00,2023-12-23 07:45:00,2023-12-28 14:02:00,2023-12-24 12:00:00,0 days 00:30:00,4 days 02:02:00
17795,10119,1511,MSC AVNI,MSC AVNI,MAVN004,2023-12-23 07:15:00,2023-12-27 07:25:00,2023-12-28 14:02:00,2023-12-28 14:00:00,4 days 00:10:00,0 days 00:02:00
20376,11437,1698,MSC ELISA XIII,MSC ELISA XIII,MEIA001,2024-03-26 18:25:00,2024-03-26 19:00:00,2024-03-29 13:10:00,2024-03-27 06:00:00,0 days 00:35:00,2 days 07:10:00
20377,11437,1703,MSC ELISA XIII,MSC ELISA XIII,MEIA002,2024-03-26 18:25:00,2024-03-28 07:36:00,2024-03-29 13:10:00,2024-03-29 13:00:00,1 days 13:11:00,0 days 00:10:00
21827,12097,1793,MSC KAYLEY,MSC KAYLEY,MKLY002,2024-05-14 03:36:00,2024-05-14 04:12:00,2024-05-19 00:32:00,2024-05-15 03:00:00,0 days 00:36:00,3 days 21:32:00
21828,12097,1800,MSC KAYLEY,MSC KAYLEY,MKLY003,2024-05-14 03:36:00,2024-05-17 16:42:00,2024-05-19 00:32:00,2024-05-19 00:00:00,3 days 13:06:00,0 days 00:32:00
22130,12253,1813,MSC MAGNUM VII,MSC MAGNUM VII,MMGG002,2024-05-25 06:20:00,2024-05-25 06:36:00,2024-05-29 20:05:00,2024-05-25 21:00:00,0 days 00:16:00,3 days 23:05:00
22131,12253,1820,MSC MAGNUM VII,MSC MAGNUM VII,MMGG003,2024-05-25 06:20:00,2024-05-29 00:06:00,2024-05-29 20:05:00,2024-05-29 20:00:00,3 days 17:46:00,0 days 00:05:00


In [58]:
경계허용=비교[(비교['접안']>=비교['입항']-pd.Timedelta(minutes=30)) & (비교['터미널출항']<=비교['출항']+pd.Timedelta(minutes=30)) & (비교['터미널출항']>비교['접안'])].copy()
경계허용수=경계허용.groupby('rid')['pid'].nunique()
분할대상번호=경계허용수[경계허용수>=2].index
분할대상=경계허용[경계허용['rid'].isin(분할대상번호)].sort_values(['rid','접안']).copy()
print('분할 전:',len(분할대상번호),'분할 후:',len(분할대상))
display(경계허용수.loc[분할대상번호])
display(분할대상[확인열].head(5))

분할 전: 16 분할 후: 32


rid
853      2
2576     2
3014     2
5989     2
8946     2
9000     2
10119    2
11032    2
11437    2
12097    2
12253    2
14496    2
16314    2
16926    2
17143    2
19262    2
Name: pid, dtype: int64

,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이
1117,853,135,XIN BIN ZHOU,XIN BIN ZHOU,MXBZ005,2022-03-05 19:48:00,2022-03-05 20:20:00,2022-03-09 23:58:00,2022-03-06 15:00:00,0 days 00:32:00,3 days 08:58:00
1118,853,143,XIN BIN ZHOU,XIN BIN ZHOU,MXBZ006,2022-03-05 19:48:00,2022-03-09 14:00:00,2022-03-09 23:58:00,2022-03-10 00:00:00,3 days 18:12:00,0 days 00:02:00
3119,2576,384,MSC MUNDRA,MSC MUNDRA,MMDU001,2022-07-09 17:00:00,2022-07-09 17:05:00,2022-07-15 11:52:00,2022-07-10 17:00:00,0 days 00:05:00,4 days 18:52:00
3120,2576,390,MSC MUNDRA,MSC MUNDRA,MMDU002,2022-07-09 17:00:00,2022-07-13 22:05:00,2022-07-15 11:52:00,2022-07-15 12:00:00,4 days 05:05:00,0 days 00:08:00
3758,3014,446,MSC LA SPEZIA,MSC LA SPEZIA,MLPZ001,2022-08-11 00:15:00,2022-08-11 00:30:00,2022-08-14 11:55:00,2022-08-12 00:00:00,0 days 00:15:00,2 days 11:55:00


## 8. 최종 대응 목록 구성
출항 우선 연결을 기본으로 하되, 분할 대상은 두 항차를 모두 반영합니다. 최종 2,928행, 원본 record 2,912개가 대응됩니다.

In [59]:
일반연결=출항연결[~출항연결['rid'].isin(분할대상번호)].copy()
최종연결=pd.concat([일반연결,별도연결,분할대상],ignore_index=True).drop_duplicates(['rid','pid'])
print('대응 행:',len(최종연결),'원본 record:',최종연결['rid'].nunique(),'port1:',최종연결['pid'].nunique())
display(최종연결[최종연결.duplicated('pid',keep=False)][확인열])

대응 행: 2928 원본 record: 2912 port1: 2928


,rid,pid,선명_record,선명_port1,모선항차,입항,접안,출항,터미널출항,입항차이,출항차이


## 9. 양쪽 미대응 확인
이름·시각·다중 항차 검토 후 남은 미대응을 제외 대상으로 정합니다. 기존 원본 기준 record 8건, port1 271건입니다.

In [60]:
record_제외=r[~r['rid'].isin(최종연결['rid'])].copy()
port1_제외=p[~p['pid'].isin(최종연결['pid'])].copy()
print('record 제외:',len(record_제외),'port1 제외:',len(port1_제외))
display(record_제외[['rid','선명','입항','출항','계선장소']])
display(port1_제외[['pid','선명','모선항차','접안','터미널출항','선석']])

record 제외: 8 port1 제외: 0


,rid,선명,입항,출항,계선장소
1757,1757,PACIFIC BUSAN,2022-05-10 03:13:00,2022-05-11 04:00:00,신항 1부두 2선석
5628,5628,TAMINA,2023-02-17 18:00:00,2023-02-21 15:58:00,신항 1부두 1선석
6987,6987,MSC COLETTE III,2023-05-23 05:05:00,2023-05-23 14:01:00,신항 1부두 3선석
12831,12831,RUN LONG,2024-07-06 21:15:00,2024-07-07 15:41:00,신항 1부두 1선석
13810,13810,PACIFIC TIANJIN,2024-09-15 15:00:00,2024-09-16 15:42:00,신항 1부두 3선석
15250,15250,NYK FUSHIMI,2025-01-02 02:20:00,2025-01-03 01:52:00,신항 1부두 2선석
16006,16006,MSC FLAVIA,2025-03-03 23:17:00,2025-03-06 11:58:00,신항 1부두 3선석
19103,19103,MSC CAIRO IV,2025-10-26 20:20:00,2025-10-27 17:18:00,신항 1부두 1선석


,pid,선명,모선항차,접안,터미널출항,선석


## 10. 16행 복사 → 32행 생성 → 네 열만 변경
입항일시·출항일시·계선장소·재항시간만 변경합니다. 원본의 나머지 열은 유지합니다.

In [61]:
분할행목록=[]
변경내역목록=[]
for _, 연결행 in 분할대상.iterrows():
    원본행=record.loc[연결행['rid']].copy()
    수정행=원본행.copy()
    수정행['입항일시']=str(연결행['접안'])
    수정행['출항일시']=str(연결행['터미널출항'])
    선석번호=str(연결행['선석']).split('(')[0].replace('T','').strip()
    수정행['계선장소']=f'신항 1부두 {선석번호}선석'
    수정행['재항시간']=str((연결행['터미널출항']-연결행['접안']).total_seconds()/3600)
    분할행목록.append(수정행)
    변경내역목록.append({'rid':연결행['rid'],'pid':연결행['pid'],'선명':원본행['선명'],'모선항차':연결행['모선항차'],
        '원래입항':원본행['입항일시'],'수정입항':수정행['입항일시'],
        '원래출항':원본행['출항일시'],'수정출항':수정행['출항일시'],
        '원래계선장소':원본행['계선장소'],'수정계선장소':수정행['계선장소'],
        '원래재항시간':원본행['재항시간'],'수정재항시간':수정행['재항시간']})
분할행=pd.DataFrame(분할행목록)
변경내역=pd.DataFrame(변경내역목록)
display(변경내역)


,rid,pid,선명,모선항차,원래입항,수정입항,원래출항,수정출항,원래계선장소,수정계선장소,원래재항시간,수정재항시간
0,853,135,XIN BIN ZHOU,MXBZ005,2022-03-05 19:48:00,2022-03-05 20:20:00,2022-03-09 23:58:00,2022-03-06 15:00:00,신항 1부두 1선석,신항 1부두 1선석,100.16666666666667,18.666666666666668
1,853,143,XIN BIN ZHOU,MXBZ006,2022-03-05 19:48:00,2022-03-09 14:00:00,2022-03-09 23:58:00,2022-03-10 00:00:00,신항 1부두 1선석,신항 1부두 1선석,100.16666666666667,10.0
2,2576,384,MSC MUNDRA,MMDU001,2022-07-09 17:00:00,2022-07-09 17:05:00,2022-07-15 11:52:00,2022-07-10 17:00:00,신항 1부두 2선석,신항 1부두 1선석,138.86666666666667,23.916666666666668
3,2576,390,MSC MUNDRA,MMDU002,2022-07-09 17:00:00,2022-07-13 22:05:00,2022-07-15 11:52:00,2022-07-15 12:00:00,신항 1부두 2선석,신항 1부두 2선석,138.86666666666667,37.916666666666664
4,3014,446,MSC LA SPEZIA,MLPZ001,2022-08-11 00:15:00,2022-08-11 00:30:00,2022-08-14 11:55:00,2022-08-12 00:00:00,신항 1부두 3선석,신항 1부두 3선석,83.66666666666667,23.5
5,3014,452,MSC LA SPEZIA,MLPZ002,2022-08-11 00:15:00,2022-08-13 05:00:00,2022-08-14 11:55:00,2022-08-14 12:00:00,신항 1부두 3선석,신항 1부두 3선석,83.66666666666667,31.0
6,5989,860,MSC ALDI III,MADI002,2023-03-15 10:47:00,2023-03-15 11:00:00,2023-03-17 13:00:00,2023-03-16 01:00:00,신항 1부두 2선석,신항 1부두 2선석,50.21666666666667,14.0
7,5989,864,MSC ALDI III,MADI003,2023-03-15 10:47:00,2023-03-16 20:40:00,2023-03-17 13:00:00,2023-03-17 13:00:00,신항 1부두 2선석,신항 1부두 2선석,50.21666666666667,16.333333333333332
8,8946,1309,BOTANY,MBON004,2023-10-01 23:20:00,2023-10-01 23:40:00,2023-10-07 07:54:00,2023-10-02 21:00:00,신항 1부두 3선석,신항 1부두 2선석,128.56666666666666,21.333333333333332
9,8946,1323,BOTANY,MBON005,2023-10-01 23:20:00,2023-10-07 00:00:00,2023-10-07 07:54:00,2023-10-07 08:00:00,신항 1부두 3선석,신항 1부두 3선석,128.56666666666666,8.0


In [62]:
유지할열=[열 for 열 in record.columns if 열 not in ['입항일시','출항일시','계선장소','재항시간']]
복사전=record.loc[분할대상['rid'].tolist(),유지할열].reset_index(drop=True)
복사후=분할행[유지할열].reset_index(drop=True)
print('나머지 열 동일:',복사전.equals(복사후))

나머지 열 동일: True


## 11. record 최종본 구성
기존 분할 대상 16행과 미대응 8행을 빼고 32행을 추가합니다. 다른 부두와 일반 대응 행의 값은 그대로입니다.

In [63]:
삭제할번호=분할대상['rid'].tolist()+record_제외['rid'].tolist()
record_최종=pd.concat([record.drop(index=삭제할번호),분할행],ignore_index=True)
record_최종['정렬용입항']=pd.to_datetime(record_최종['입항일시'])
record_최종=record_최종.sort_values('정렬용입항').drop(columns='정렬용입항').reset_index(drop=True)
record_최종_1부두=record_최종[record_최종['계선장소'].str.contains('신항 1부두')]
print('전체:',len(record_최종),'1부두:',len(record_최종_1부두))
display(record_최종_1부두.head(5))

전체: 19989 1부두: 2928


,호출부호,선명,입항횟수,총톤수,입항일시,출항일시,CIQ수속일자,수리일시,MRN 번호,계선장소,차항지,전출항지,재항시간
3,D5TL4,MSC JEWEL,2022.0,"140,976",2022-01-01 03:22:00,2022-01-03 05:00:00,2022-01-03 5:00,2022-01-03 11:19,22MSCU0004E,신항 1부두 3선석,MANZANILLO,HONG KONG,49.63333333333333
7,VRLY8,COSCO SAO PAULO,2022.0,"40,465",2022-01-01 07:10:00,2022-01-02 06:55:00,2022-01-02 6:55,2022-01-03 11:18,22COSU0024E,신항 1부두 1선석,ENSENADA,QINGDAO,23.75
17,D5HE7,COPIAPO,2022.0,"93,685",2022-01-02 08:15:00,2022-01-04 12:05:00,2022-01-04 12:05,2022-01-05 10:24,22HLCU4204E,신항 1부두 2선석,MANZANILLO,SHANGHAI,51.833333333333336
18,3EVH7,MSC ELAINE,2022.0,"95,681",2022-01-02 08:58:00,2022-01-03 19:55:00,2022-01-03 20:00,2022-01-04 10:27,22MSCU0051E,신항 1부두 1선석,TACOMA,SHANGHAI,34.95
30,CQZO,MSC DESIREE,2022.0,"97,805",2022-01-03 07:10:00,2022-01-04 20:20:00,2022-01-04 20:00,2022-01-05 10:27,22MSCU0076E,신항 1부두 3선석,SHANGHAI,SINGAPORE,37.166666666666664


## 12. port1 최종본 구성 → 건수 확인
연결된 원본 port1 행만 추출.

In [64]:
port1_최종=port1.loc[sorted(최종연결['pid'].unique())].copy().reset_index(drop=True)
display(pd.DataFrame({'자료':['record 전체','record 1부두','port1'],'행 수':[len(record_최종),len(record_최종_1부두),len(port1_최종)]}))
print('1부두 행 수 동일:',len(record_최종_1부두)==len(port1_최종))
display(port1_최종.head(5))

,자료,행 수
0,record 전체,19989
1,record 1부두,2928
2,port1,2928


1부두 행 수 동일: True


,선석,선사,모선항차,선사항차,선명,ROUTE,반입마감시한,접안(예정)일시,출항(예정)일시,양하,적하,Shift
0,T3(P),MSC,MJEW008,FA147A/FA147A,MSC JEWEL,ASPA2E,2021-12-31 16:00,2022-01-01 3:40,2022-01-03 5:00,1263,2905,2
1,T1(P),COS,COSP001,090E/090E,COSCO SAO PAULO,WSA3,2021-12-31 20:00,2022-01-01 7:20,2022-01-02 7:00,402,1066,0
2,T2(P),HLC,COPX001,2140W/2152E,COPIAPO,AN1,2022-01-01 22:00,2022-01-02 8:10,2022-01-04 12:00,1471,1998,108
3,T1(P),MSC,MELN003,UK150A/UK150A,MSC ELAINE,CHINK,2022-01-01 22:00,2022-01-02 9:10,2022-01-03 20:00,838,1281,2
4,T3(P),MSC,MDES002,FI144R/FI150A,MSC DESIREE,AS2,2022-01-02 19:00,2022-01-03 7:20,2022-01-04 20:00,1139,1900,570


## 13. 저장
원본을 덮어쓰지 않도록 별도 폴더에 저장합니다.
접안시간 겹침 항차는 삭제하지 않습니다.
대응 목록의 rid·pid는 수정 전 원본의 행 번호입니다.
예상 건수와 다르거나 중복 연결이 출력되면 해당 단계부터 확인하세요.
필요시 주석 해제 후 사용할것.

In [65]:
# OUT=DATA/'1부두_전처리완료'
# OUT.mkdir(exist_ok=True)
# record_최종.to_csv(OUT/'record_except_168.csv',index=False,encoding='utf-8-sig')
# port1_최종.to_csv(OUT/'port1.csv',index=False,encoding='utf-8-sig')
# 변경내역.to_csv(OUT/'분할16건_32행_변경내역.csv',index=False,encoding='utf-8-sig')
# record_제외[record.columns].to_csv(OUT/'record_미대응8건.csv',index=False,encoding='utf-8-sig')
# port1_제외[port1.columns].to_csv(OUT/'port1_미대응271건.csv',index=False,encoding='utf-8-sig')
# 최종연결[확인열].to_csv(OUT/'최종_항차대응목록.csv',index=False,encoding='utf-8-sig')
# print('저장 위치:',OUT.resolve())

저장 위치: C:\work\final_project\data\old\1부두_전처리완료
